# M00 — 為什麼需要 LangGraph

本 notebook 對應 `README.md`，逐格執行即可。

第二冊開場。這一格課的重點不是 API，而是**心智轉換**：
從第一冊「單向、無狀態的 LCEL 管線」，升級到「有狀態的圖（state + node + edge）」。

我們會：

1. 用一個情境（草稿 → 批改 → 沒達標就重寫）親眼看到純 LCEL 卡在哪。
2. 用文字畫出 state / node / edge 心智圖。
3. 跑一個極簡的 LangGraph「hello world」當 teaser（細節留給 M01）。

## 1. 環境準備

沿用整套課程的共用 helper，讓範例與供應商無關（OpenAI / Anthropic / Ollama 皆可）。
這格載入 `get_model`，後面情境示範會用到。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 回顧：第一冊的 LCEL chain 是一條「單向輸送帶」

先重溫第一冊的寫法。`prompt | model | parser` 把組件串成管線：
資料從左進、右出，中間每個組件只被「經過一次」。

預期輸出：一段文字草稿（實際內容依模型而定，重點在「結構」不在「字句」）。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

write_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是文案寫手，請寫一段約 50 字的產品標語草稿。"),
    ("human", "產品：{product}"),
])

# A classic LCEL pipeline: single direction, stateless.
write_chain = write_prompt | model | StrOutputParser()

draft = write_chain.invoke({"product": "降噪耳機"})
print(draft)
# Expected output: 一段約 50 字的標語草稿文字。

## 3. 痛點登場：流程需要「迴圈」時，chain 就彆扭了

現在把需求變真實一點：

> 寫草稿 → 批改 → **沒達標就重寫** → 再批改 …… 直到達標（最多 3 次）。

這裡有三個 chain 做不到的東西：

- **迴圈**：「沒達標就回去重寫」要往回走，但管線是單向的。
- **狀態**：要記住「第幾次了」「上一輪被批評什麼」，但每次 `invoke` 都是乾淨、互不認識的。
- **分支**：「達標就停、否則繼續」是一個依結果決定的岔路。

結果你只能在管線**外面**手寫一個 `while` 迴圈、自己用變數存中間狀態、自己塞回去。
邏輯散落在 LCEL 之外——這就是訊號：**該換工具了。**

下面這格故意用「純 LCEL + 外部 while」把這份彆扭寫出來給你看。

In [ ]:
# This is the AWKWARD way: LCEL can't loop, so we bolt a while-loop on the outside
# and juggle state by hand. Read it as a "smell", not as good practice.
review_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你是嚴格的文案主編。看完標語草稿後，"
     "若夠好就只回覆「PASS」；否則回覆一句具體修改建議。"),
    ("human", "草稿：{draft}"),
])
review_chain = review_prompt | model | StrOutputParser()

draft = write_chain.invoke({"product": "降噪耳機"})

# All this bookkeeping lives OUTSIDE the pipeline — that's the problem.
max_rounds = 3
for round_no in range(1, max_rounds + 1):
    review = review_chain.invoke({"draft": draft})
    if "PASS" in review.upper():
        print(f"第 {round_no} 輪達標：{draft}")
        break
    # Feed the critique back in by hand, then loop again.
    draft = write_chain.invoke({"product": f"降噪耳機（請依建議修改：{review}）"})
else:
    print(f"用完 {max_rounds} 輪仍未達標，最後草稿：{draft}")
# Expected output: 幾輪後印出達標草稿，或用完輪數的最後草稿。
# 注意：迴圈、計數、把 review 塞回 prompt——全都在 LCEL 外面手刻，這就是痛點。

## 4. 心智模型轉換：把「流程」看成一張「圖」

LangGraph 的解法是：別在管線外手刻迴圈，直接把整個流程畫成一張**圖**。
只要記住三個詞：

- **State**：一個共享的資料結構（白板），每個節點都能讀、能寫。
- **Node**：一個普通函式 `f(state) -> dict`，讀白板、做事、把結果寫回白板。
- **Edge**：節點之間的路標，決定「做完這步換做哪步」；可固定，也可依狀態決定。

再加 `START`（入口）與 `END`（出口）兩個特殊節點。

把第 3 節的痛點翻成圖，心智圖長這樣（這一格純文字，不執行）：

```text
        ┌─────────────────────────────┐
        │   State（共享白板）          │
        │   draft:   目前草稿           │
        │   review:  上一輪批改意見      │
        │   passed:  是否達標 (bool)     │
        └─────────────────────────────┘

   START ──▶ write ──▶ review ──▶ (passed?) ──否──▶ write   ← 迴圈！往回走
                                      │
                                     是
                                      ▼
                                     END
```

對照第 3 節：原本散在 `while` 裡的「計數、暫存、塞回去」，現在都收進 **State**；
「沒達標就回去重寫」那條**迴圈邊**，圖天生就能表達。
關鍵體會（會貫穿整個第二冊）：**先把 State 設計對，流程就會變簡單。**

## 🧪 練習一：把流程翻成 state / node / edge

想像一個「客服助理」流程：

> 收到問題 → 先查 FAQ → 查得到就直接回答；查不到就轉真人客服。

不要寫程式，只用紙筆或註解回答三件事：

1. **State** 該放哪些欄位？（提示：問題本身、FAQ 命中與否、最後答覆……）
2. 有哪些 **node**？各做一件什麼事？
3. 哪條 **edge** 是「依狀態決定」的分支？它根據 State 的哪個欄位來分？

把答案寫在下面這格的註解裡。

In [ ]:
# 你的答案（用註解寫）：
# State 欄位：
#   - ...
# Nodes：
#   - ...
# 分支 edge 依據的欄位：
#   - ...

## 5. Teaser：一個極簡的 LangGraph「hello world」

先別管細節，感受一下「畫圖」是什麼手感。這是最小的一張圖：
一個 State、一個 node、`START → node → END`，然後 `compile`、`invoke`。

重點先看流程，不用背 API——**M01 會把每一行正式拆給你看。**

預期輸出：`{'message': 'hello, LangGraph!'}`

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


# 1) State: the shared "whiteboard" for the whole graph.
class State(TypedDict):
    message: str


# 2) Node: a plain function. Read state, do work, return ONLY the fields to update.
def say_hello(state: State) -> dict:
    return {"message": "hello, LangGraph!"}


# 3) Wire it up: START -> say_hello -> END, then compile into a runnable graph.
builder = StateGraph(State)
builder.add_node("say_hello", say_hello)
builder.add_edge(START, "say_hello")
builder.add_edge("say_hello", END)
graph = builder.compile()

# 4) Run it just like anything else in the course: .invoke(...)
result = graph.invoke({"message": ""})
print(result)
# Expected output: {'message': 'hello, LangGraph!'}

## 6. 看一眼這張圖長什麼樣

`compile()` 後的圖可以畫出來。文字版最方便，不需要額外套件。

預期輸出：一張 ASCII 流程圖，能看到 `__start__ → say_hello → __end__` 的走向。

In [ ]:
# Render the graph as text so you can see the flow you just built.
print(graph.get_graph().draw_ascii())
# Expected output: ASCII 流程圖，從 __start__ 經 say_hello 到 __end__。

## 🧪 練習二：在圖裡加第二個 node

以第 5 節的 `graph` 為基礎（**只在腦中或註解推演即可，本模組不要求跑成功**）：

1. 想像再加一個 node `shout`，把 `message` 變成大寫（`state["message"].upper()`）。
2. 把流程改成 `START → say_hello → shout → END`。
3. 預測 `graph.invoke({"message": ""})` 會印出什麼？

把你的預測寫在下面註解。真正動手 `add_node` / `add_edge` 的細節，M01 會正式教。

In [ ]:
# 你的預測（用註解寫）：
# 加上 shout 之後，最終 message 會是：
#   - ...

## 小結 & 下一步

這一格課你應該帶走的：

- **chain 是單向、無狀態的輸送帶**；一旦流程需要迴圈、分支、記憶或人介入，它就到頂了
  （第 3 節那個「外面手刻 while」就是訊號）。
- **LangGraph 用 state + node + edge 描述流程**，迴圈與分支變成圖的自然能力；
  先把 **State 設計對**，流程就簡單。
- chain → `create_agent` → 自畫 LangGraph，是一條「掌控度由低到高」的光譜；
  第一冊的 `create_agent` 本身就是一張現成的圖。

**下一步：M01 — StateGraph 基礎。** 我們會正式拆解 `StateGraph`、`add_node`、
`add_edge`、`START` / `END`，把今天這張 teaser 圖一行一行講清楚，並讓你從零畫出
自己的第一張能跑的圖。